In [0]:
%sql
-- 1. Criação do schema legado no Hive Metastore
CREATE SCHEMA IF NOT EXISTS hive_metastore.legado_financas;

-- 2. Tabela de clientes com dados sensíveis (LGPD: CPF, e-mail)
CREATE TABLE IF NOT EXISTS hive_metastore.legado_financas.clientes (
    id_cliente INT,
    nome STRING,
    cpf STRING,
    email STRING,
    renda_mensal DOUBLE,
    data_cadastro DATE
) USING DELTA;

-- 3. Inserindo dados simulados
INSERT INTO hive_metastore.legado_financas.clientes VALUES
(1, 'Carlos Silva', '123.456.789-00', 'carlos.silva@email.com', 4500.00, '2023-01-15'),
(2, 'Mariana Costa', '987.654.321-11', 'mariana.costa@email.com', 8200.00, '2023-03-22'),
(3, 'Joao Santos', '456.789.123-22', 'joao.santos@email.com', 3100.00, '2023-06-10');

-- 4. Tabela de transações de crédito
CREATE TABLE IF NOT EXISTS hive_metastore.legado_financas.transacoes_credito (
    id_transacao STRING,
    id_cliente INT,
    valor_transacao DOUBLE,
    status STRING,
    data_transacao TIMESTAMP
) USING DELTA;

INSERT INTO hive_metastore.legado_financas.transacoes_credito VALUES
('TX-1001', 1, 150.50, 'APROVADA', current_timestamp()),
('TX-1002', 2, 1200.00, 'APROVADA', current_timestamp()),
('TX-1003', 1, 3500.00, 'NEGADA', current_timestamp());

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7745272384787376>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- 1. Criação do schema legado no Hive Metastore\nCREATE SCHEMA IF NOT EXISTS hive_metastore.legado_financas;\n\n-- 2. Tabela de clientes com dados sensíveis (LGPD: CPF, e-mail)\nCREATE TABLE IF NOT EXISTS hive_metastore.legado_financas.clientes (\n    id_cliente INT,\n    nome STRING,\n    cpf STRING,\n    email STRING,\n    renda_mensal DOUBLE,\n    data_cadastro DATE\n) USING DELTA;\n\n-- 3. Inserindo dados simulados\nINSERT INTO hive_metastore.legado_financas.clientes VALUES\n(1, 'Carlos Silva', '123.456.789-00', 'carlos.silva@email.com', 4500.00, '2023-01-15'),\n(2, 'Mariana Costa', '987.654.321-11', 'mariana.costa@email.com', 8200.00, '2023-03-22'),\n(3, 'Joao Santos', '456.789.123-22', 'joao.santos@email.com', 3100.00, '2023-06-10');\n\n-- 4.

In [0]:
%sql
-- 1. Criação do schema de demonstração dentro do catálogo 'workspace'
CREATE SCHEMA IF NOT EXISTS workspace.legado_financas;

-- 2. Tabela de clientes com dados sensíveis (LGPD: CPF, e-mail)
CREATE TABLE IF NOT EXISTS workspace.legado_financas.clientes (
    id_cliente INT,
    nome STRING,
    cpf STRING,
    email STRING,
    renda_mensal DOUBLE,
    data_cadastro DATE
) USING DELTA;

-- 3. Inserindo dados simulados
INSERT INTO workspace.legado_financas.clientes VALUES
(1, 'Carlos Silva', '123.456.789-00', 'carlos.silva@email.com', 4500.00, '2023-01-15'),
(2, 'Mariana Costa', '987.654.321-11', 'mariana.costa@email.com', 8200.00, '2023-03-22'),
(3, 'Joao Santos', '456.789.123-22', 'joao.santos@email.com', 3100.00, '2023-06-10');

-- 4. Tabela de transações de crédito
CREATE TABLE IF NOT EXISTS workspace.legado_financas.transacoes_credito (
    id_transacao STRING,
    id_cliente INT,
    valor_transacao DOUBLE,
    status STRING,
    data_transacao TIMESTAMP
) USING DELTA;

INSERT INTO workspace.legado_financas.transacoes_credito VALUES
('TX-1001', 1, 150.50, 'APROVADA', current_timestamp()),
('TX-1002', 2, 1200.00, 'APROVADA', current_timestamp()),
('TX-1003', 1, 3500.00, 'NEGADA', current_timestamp());

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
-- 1. Criar o schema da nossa camada de dados refinada (Silver)
CREATE SCHEMA IF NOT EXISTS workspace.silver_financas;

-- 2. Migrar os dados de clientes para a camada Silver
CREATE OR REPLACE TABLE workspace.silver_financas.clientes 
AS SELECT * FROM workspace.legado_financas.clientes;

-- 3. Criar a função de mascaramento para o CPF
-- (Se o usuário não for do grupo 'admin', o CPF é automaticamente ocultado)
CREATE OR REPLACE FUNCTION workspace.silver_financas.mask_cpf(cpf STRING)
RETURN IF(
    is_account_group_member('admin'),
    cpf,
    CONCAT('***.', SUBSTRING(cpf, 5, 3), '.***-', SUBSTRING(cpf, 10, 2))
);

-- 4. Aplicar o mascaramento diretamente na coluna CPF da tabela Silver
ALTER TABLE workspace.silver_financas.clientes 
ALTER COLUMN cpf SET MASK workspace.silver_financas.mask_cpf;

In [0]:
%sql
SELECT id_cliente, nome, cpf, email, renda_mensal 
FROM workspace.silver_financas.clientes;

id_cliente,nome,cpf,email,renda_mensal
1,Carlos Silva,***.456.***-89,carlos.silva@email.com,4500.0
2,Mariana Costa,***.654.***-21,mariana.costa@email.com,8200.0
3,Joao Santos,***.789.***-23,joao.santos@email.com,3100.0
